## N-Grams and Language Models

**Author:** Diego Besada

Created using [Pierre Nugues' template](https://github.com/pnugues/edan20/blob/master/labs_2026/2-language_models.ipynb)

In [1]:
import math, json, os, requests, regex as re
import pandas as pd

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

#### Building the n-gram models

We start by downloading the *Selma Lagerlöf* corpus and loading it as a single string. To inspect the text and later sanity-check the tokenizer, we run a small concordance around the phrase *Nils Holgersson*.

In [2]:
SELMA_URL = 'https://github.com/pnugues/ilppp/blob/master/programs/corpus/Selma.txt?raw=true'

if not os.path.isfile('Selma.txt'):
    req = requests.get(SELMA_URL)
    with open('Selma.txt', 'wb') as f:
        f.write(req.content)

corpus = open('Selma.txt', 'r', encoding='utf-8').read()

test_para = 'En  gång hade de på Mårbacka en barnpiga, som hette Back-Kajsa. \
Hon var nog sina tre alnar lång, hon hade ett stort, grovt ansikte med stränga, mörka drag, \
hennes händer voro hårda och fulla av sprickor, som barnens hår fastnade i, \
när hon kammade dem, och till humöret var hon dyster och sorgbunden.'

In [3]:
pattern = 'Nils Holgersson'
width = 25

clean_corpus = re.sub(r'\s+', ' ', corpus).strip()
pattern = re.escape(pattern).replace(r'\ ', r'\s+')

concordance = rf'.{{0,{width}}}{pattern}.{{0,{width}}}'

iterator = re.finditer(concordance, clean_corpus)
for _ in range(5):
    next(iterator, None).group()

'Selma Lagerlöf Nils Holgerssons underbara resa genom Sv'

'! Se på Tummetott! Se på Nils Holgersson Tummetott!» Genast vände'

'r,» sade han. »Jag heter Nils Holgersson och är son till en husma'

'lden. »Inte är det värt, Nils Holgersson, att du är ängslig eller'

' i dem. På den tiden, då Nils Holgersson drog omkring med vildgäs'

`tokenize` splits the text into a list of word tokens using the regular expression `\p{L}+`, which matches one or more Unicode letters. Punctuation, digits and whitespace are dropped; the corpus yields 923,485 tokens, 44,256 of them unique.

In [4]:
def tokenize(text: str) -> list:
    return re.findall(r'\p{L}+', text)

words = tokenize(corpus)

'words', len(words)
'unique words', len(set(words))
'unique words (lowercase)', len(set(word.lower() for word in words))

('words', 923485)

('unique words', 44256)

('unique words (lowercase)', 41032)

`clean` normalizes the raw text before segmenting: it replaces every character that is not a letter or sentence punctuation (`. ; : ? !`) with a space and collapses repeated spaces.

In [5]:
def clean(text: str) -> str:
    '''
    Tokenizes words
    '''
    non_letter = r'[^\p{L}.;:?!]'
    text = re.sub(non_letter, ' ', text)
    text = re.sub(r' +', ' ', text)
    return text

clean(test_para)

'En gång hade de på Mårbacka en barnpiga som hette Back Kajsa. Hon var nog sina tre alnar lång hon hade ett stort grovt ansikte med stränga mörka drag hennes händer voro hårda och fulla av sprickor som barnens hår fastnade i när hon kammade dem och till humöret var hon dyster och sorgbunden.'

`segment_sentences` marks sentence boundaries by inserting `</s>` and `<s>` between a punctuation mark and the following capital letter, wraps the whole text in `<s>` ... `</s>`, removes the punctuation, collapses whitespace and lowercases everything.

In [6]:
def segment_sentences(text: str) -> str:
    '''
    Segments sentences
    '''
    sentence_boundaries = r'([.;:?!])\s+(\p{Lu})'
    sentence_markup = r'\1 </s>\n<s> \2'
    text = re.sub(sentence_boundaries, sentence_markup, text)
    text = '<s> ' + text + ' </s>'
    text = re.sub(r'[.;:?!]', '', text)
    text = re.sub(r'\p{Zs}+', ' ', text)
    return text.lower()

print(segment_sentences(test_para))

<s> en gång hade de på mårbacka en barnpiga, som hette back-kajsa </s>
<s> hon var nog sina tre alnar lång, hon hade ett stort, grovt ansikte med stränga, mörka drag, hennes händer voro hårda och fulla av sprickor, som barnens hår fastnade i, när hon kammade dem, och till humöret var hon dyster och sorgbunden </s>


`unigrams` counts how often each token appears; the start marker `<s>` occurs 59,047 times, i.e. once per sentence. `unigram_lm` then scores a sentence as the product of the independent word probabilities $P(w_i) = C(w_i)/N$ and reports the geometric mean, the entropy rate and the perplexity.

In [7]:
def unigrams(words: list) -> dict:
    '''
    Counts the frequency of each word in a list of words
    '''
    frequency = {}
    for i in range(len(words)):
        if words[i] in frequency:
            frequency[words[i]] += 1
        else:
            frequency[words[i]] = 1
    return frequency

words = re.split(r'\s+', segment_sentences(clean(corpus)).strip())
frequency = unigrams(words)
list(frequency.items())[:5]

[('<s>', 59047),
 ('selma', 52),
 ('lagerlöf', 270),
 ('nils', 87),
 ('holgerssons', 6)]

In [8]:
def unigram_lm(frequency, sent_words) -> tuple[pd.DataFrame, dict[str, float]]:
    '''
    Computing the sentence probs with a unigram model.
    '''

    nbr_words = sum(frequency.values())
    prob_unigrams = 1.0
    for word in sent_words:
        count = frequency.get(word, 0)
        prob = count / nbr_words
        prob_unigrams *= prob

    geometric_mean = prob_unigrams ** (1 / len(sent_words))
    entropy_rate = -math.log2(prob_unigrams) / len(sent_words)
    perplexity = 2 ** entropy_rate

    return (
        pd.DataFrame({
            'word': sent_words,
            'count': [frequency.get(word, 0) for word in sent_words],
            'prob': [frequency.get(word, 0) / nbr_words for word in sent_words]
        }).set_index('word'),
        {
            'prob_unigrams': prob_unigrams,
            'geometric_mean': geometric_mean,
            'entropy_rate': entropy_rate,
            'perplexity': perplexity
        }
    )

sentence = 'det var en gång en katt som hette nils </s>'.split()
unigram_df, unigram_stats = unigram_lm(frequency, sentence)
unigram_df
unigram_stats

,count,prob
word,,
det,21108,0.020266
var,12090,0.011608
en,13514,0.012975
gång,1332,0.001279
en,13514,0.012975
katt,16,0.000015
som,16288,0.015638
hette,97,0.000093
nils,87,0.000084


{'prob_unigrams': 5.3651155337425844e-27,
 'geometric_mean': 0.0023602494649885993,
 'entropy_rate': 8.726844932328587,
 'perplexity': 423.68402782577465}

`bigrams` counts the adjacent word pairs. `bigrams_lm` conditions each word on the previous one, $P(w_i | w_{i-1}) = C(w_{i-1}, w_i)/C(w_{i-1})$; when a bigram is unseen it backs off to the unigram probability $C(w_i)/N$. `<s>` is prepended only to condition the first word, so the sentence has $n = len(sent\_words) - 1$ terms.

In [9]:
def bigrams(words: list) -> dict:
    '''
    Counts the frequency of each bigram in a list of words
    '''
    frequency = {}
    for i in range(len(words) - 1):
        bigram = (words[i], words[i + 1])
        if bigram in frequency:
            frequency[bigram] += 1
        else:
            frequency[bigram] = 1
    return frequency

words = re.split(r'\s+', segment_sentences(clean(corpus)).strip())
frequency_bigrams = bigrams(words)
list(frequency_bigrams.items())[:5]

[(('<s>', 'selma'), 8),
 (('selma', 'lagerlöf'), 11),
 (('lagerlöf', 'nils'), 1),
 (('nils', 'holgerssons'), 6),
 (('holgerssons', 'underbara'), 4)]

In [10]:
def bigrams_lm(frequency: dict[str, int], frequency_bigrams: dict[tuple[str, str], int], sent_words: list[str]) -> tuple[pd.DataFrame, dict[str, float]]:
    '''
    Computing the sentence probs with a bigram model.
    Backs off to the unigram probability when the bigram count is 0.
    '''

    sent_words = ['<s>'] + sent_words
    nbr_words = sum(frequency.values())
    prob_bigrams = 1.0
    for i in range(len(sent_words) - 1):
        word1, word2 = sent_words[i], sent_words[i + 1]
        count_bigram = frequency_bigrams.get((word1, word2), 0)
        count_word1 = frequency.get(word1, 0)
        if count_bigram == 0:
            # Backoff to the unigram probability
            prob_bigram = frequency.get(word2, 0) / nbr_words
        else:
            prob_bigram = count_bigram / count_word1
        prob_bigrams *= prob_bigram

    nbr_bigrams = len(sent_words) - 1
    geometric_mean = prob_bigrams ** (1 / nbr_bigrams)
    entropy_rate = -math.log2(prob_bigrams) / nbr_bigrams
    perplexity = 2 ** entropy_rate

    return (
        pd.DataFrame({
            'bigram': [(sent_words[i], sent_words[i + 1]) for i in range(len(sent_words) - 1)],
            'count': [frequency_bigrams.get((sent_words[i], sent_words[i + 1]), 0) for i in range(len(sent_words) - 1)],
            'prob': [
                frequency_bigrams.get((sent_words[i], sent_words[i + 1]), 0) / frequency.get(sent_words[i], 0)
                if frequency_bigrams.get((sent_words[i], sent_words[i + 1]), 0) > 0
                else frequency.get(sent_words[i + 1], 0) / nbr_words
                for i in range(len(sent_words) - 1)
            ]
        }).set_index('bigram'),
        {
            'prob_bigrams': prob_bigrams,
            'geometric_mean': geometric_mean,
            'entropy_rate': entropy_rate,
            'perplexity': perplexity
        }
    )

sentence = 'det var en gång en katt som hette nils </s>'.split()
bigram_df, bigram_stats = bigrams_lm(frequency, frequency_bigrams, sentence)
bigram_df
bigram_stats

,count,prob
bigram,,
"(<s>, det)",5672,0.096059
"(det, var)",3839,0.181874
"(var, en)",712,0.058892
"(en, gång)",706,0.052242
"(gång, en)",20,0.015015
"(en, katt)",6,0.000444
"(katt, som)",2,0.125000
"(som, hette)",45,0.002763
"(hette, nils)",0,0.000084


{'prob_bigrams': 2.376169768780815e-19,
 'geometric_mean': 0.013727382866049192,
 'entropy_rate': 6.186799588766881,
 'perplexity': 72.84709764111099}

`trigrams` counts contiguous triples of tokens.

In [11]:
def trigrams(words: list) -> dict:
    '''
    Counts the frequency of each trigram in a list of words
    '''
    frequency = {}
    for i in range(len(words) - 2):
        trigram = (words[i], words[i + 1], words[i + 2])
        if trigram in frequency:
            frequency[trigram] += 1
        else:
            frequency[trigram] = 1
    return frequency

frequency_trigrams = trigrams(words)

Finally, we export each n-gram dictionary to a JSONL file, one object per line with the `ngram` and its `count`.

In [12]:
def export_ngrams_jsonl(dictionary, file_name):
    with open(file_name, "w", encoding="utf-8") as f:
        for k, v in dictionary.items():
            line = {"ngram": [k] if isinstance(k, str) else list(k),
                    "count": v}
            f.write(json.dumps(line, ensure_ascii=False) + "\n")

export_ngrams_jsonl(frequency, "unigrams.jsonl")
export_ngrams_jsonl(frequency_bigrams, "bigrams.jsonl")
export_ngrams_jsonl(frequency_trigrams, "trigrams.jsonl")

#### Online prediction of the next word

We start writing a text, the most probable first word given that we have tiped `de` is: 

In [13]:
CANDIDATES = 5

In [14]:
starting_text = 'de'

candidates = {bigram: count
              for bigram, count in frequency_bigrams.items()
              if bigram[0] == '<s>' and bigram[1].startswith(starting_text)}
sorted_candidates = sorted(candidates.items(), key=lambda x: (-x[1], x[0][1]))

[bigram[1] for bigram, _ in sorted_candidates[:CANDIDATES]]

['det', 'de', 'den', 'detta', 'denna']

Now we suppose that the user has typed: `Det var en`. After detecting a space, the program starts predicting a next possible word:

In [15]:
current_text = 'det var en'
tokens = tokenize(current_text)

candidates = {trigram: count
              for trigram, count in frequency_trigrams.items()
              if list(trigram[:2]) == tokens[-2:]}
sorted_candidates = sorted(candidates.items(), key=lambda x: (-x[1], x[0][2]))
[trigram[2] for trigram, _ in sorted_candidates[:CANDIDATES]]

['stor', 'liten', 'gammal', 'god', 'sådan']

Finally, we suppose that the user has typed `Det var en g_`. We will rank the five possible candidates:

In [16]:
tokens = tokenize(current_text)

candidates = {trigram: count
              for trigram, count in frequency_trigrams.items()
              if list(trigram[:2]) == tokens[-3:-1] and trigram[2].startswith(tokens[-1])}
sorted_candidates = sorted(candidates.items(), key=lambda x: (-x[1], x[0][2]))
[trigram[2] for trigram, _ in sorted_candidates[:CANDIDATES]]

['en', 'endast', 'ensamt']